# vl6 Two-Stage Recommendation Model

## Data Aquisition

First, we need to make a CSR matrix where the rows are playlists and the columns are the songs in that playlist. This means we'll need to keep a running list of which songs are already in the matrix so one song isn't logged in multiple different columns. 

In [3]:
import json
from pathlib import Path
from scipy import sparse
from tqdm.auto import tqdm

SEED = 48635

# Load slices to memory
spotify_mpd = Path("spotify_million_playlist_dataset")
playlists_path = spotify_mpd / "data"
slice_paths = list(playlists_path.glob(r"*.json"))

# and build the matrix
tracks = {}
rows, cols, data = [], [], []

row = 0
for spth in tqdm(slice_paths, desc = "slices"):
    with open(spth, 'r') as f:
        slice = json.load(f)
        for plist in slice["playlists"]:
            songs_seen = set()
            for track in plist["tracks"]:
                col = tracks.setdefault(track["track_uri"], len(tracks))
                if col in songs_seen:
                    continue
                songs_seen.add(col)
                rows.append(row)
                cols.append(col)
                data.append(1.0)
            row += 1

slices:   0%|          | 0/1000 [00:00<?, ?it/s]

In [3]:
total_tracks = len(tracks)
total_playlists = row

plist_tracks_matrix = sparse.coo_matrix(
    (data, (rows, cols)),
    shape=(total_playlists, total_tracks),
    dtype="float32",
).tocsr()

### Initialize Test Playlist

Making a test playlist is surprisingly difficult. We need to check what songs actually exist in the training set, then find the indices of the songs in our test playlist.

#### SQL Database on Disk for Song Search

In [ ]:
import sqlite3
import json
from pathlib import Path
from tqdm.auto import tqdm

spotify_mpd = Path("spotify_million_playlist_dataset")
playlists_path = spotify_mpd / "data"
slice_paths = list(playlists_path.glob(r"*.json"))

conn = sqlite3.connect("spotify_available_tracks.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tracks (
    artist_name TEXT,
    track_name TEXT,
    raw_data TEXT,
    PRIMARY KEY (artist_name, track_name)
);
""")

for spth in tqdm(slice_paths, desc = "slices"):
    with open(spth, 'r') as f:
        slice = json.load(f)
        for plist in slice["playlists"]:
            for track in plist["tracks"]:
                cursor.execute(
                    "INSERT OR REPLACE INTO tracks VALUES (?, ?, ?)",
                    (track["artist_name"], track["track_name"], json.dumps(track))
                )

print("committing... ", end='')
conn.commit()
print('complete.')
cursor.close()
conn.close()

slices:   0%|          | 0/1000 [00:00<?, ?it/s]

committing... complete.


In [ ]:
conn = sqlite3.connect("spotify_available_tracks.db")
cursor = conn.cursor()

cursor.execute("SELECT COUNT (*) FROM tracks")
row = cursor.fetchone()
print(row)

cursor.close()
conn.close()

OperationalError: near ")": syntax error

## First Stage

### Weighted Regularized Matrix Factorization (WRMF)

In [4]:
"""
NOTE: The authors of the whitepaper on page 2:
"After parameter sweeps we found that using rank 200 with 
α = 100 and λU = λV = 0.001 produced good performance."
"""

import os

# implicit has some problems with NVIDIA drivers
try:
    import ctypes, nvidia
    _nv_libdir = os.path.join(list(nvidia.__path__)[0], "cu13", "lib")
    for _soname in ("libcudart.so.13", "libcublas.so.13", "libcublasLt.so.13",
                    "libcurand.so.10", "libnvrtc.so.13"):
        _p = os.path.join(_nv_libdir, _soname)
        if os.path.exists(_p):
            ctypes.CDLL(_p, mode=ctypes.RTLD_GLOBAL)
except Exception:
    pass


import implicit
from implicit.cpu.als import AlternatingLeastSquares as _CPUALS
from pathlib import Path
if not implicit.gpu.HAS_CUDA:
    print("CUDA not available. CPU will be used.")

WRMF_PATH = Path("wrmf_model.npz")

if WRMF_PATH.exists():
    # save() always serializes CPU-format factors, so load via the CPU class
    # then move to GPU for fast recommend/similar-items queries.
    wrmf = _CPUALS.load(WRMF_PATH)
    if implicit.gpu.HAS_CUDA:
        wrmf = wrmf.to_gpu()
    print(f"Loaded WRMF from {WRMF_PATH}")
else:
    wrmf = implicit.als.AlternatingLeastSquares(
        factors = 200,
        regularization = 0.001,
        alpha = 100,
        random_state = SEED,
    )
    wrmf.fit(plist_tracks_matrix)
    wrmf.save(WRMF_PATH)  # writes wrmf_model.npz
    print(f"Trained and saved WRMF to {WRMF_PATH}")

Loaded WRMF from wrmf_model.npz


#### Test recommendations

### Convolutional Neural Network (CNN)